# 第16章 算子优化 Agent 设计

**操作手册** | 读题、生成、评测、晋升与反思

本手册对应文档：`docs/part3-agent/chapter16/index.md`  
本手册对应代码：`code/part3-agent/`

---


## 在云端运行本章

原文中的性能数字来自 Radeon 8060S（`gfx1151`）参考运行。云端 GPU 型号不同，运行单元以当前设备输出为准。

云端已提供 ROCm、PyTorch 和 Triton；执行单元会补充 Agent 所需的 Python 第三方包。

本章实际调用 Radeon Cloud 提供的 `DeepSeek-V4-Flash`；`Qwen3.6-35B-A3B` 也可按相同方式获取接口信息。


## 本章目标、前置知识与产物

本章把第 14 章的循环骨架和第 15 章的权威工具，组装成一个完整的**算子优化 Agent**。流程不是写死的流水线，而是由 LLM 多轮驱动；真假仍由工具锁死。

学完本章，你应该能够：

- 画出「读题 → 生成 → compile → bench → profile → accept → 反思」的循环；
- 指出每一步在源码中的落点；
- 解释为什么晋升必须有配对证据，而不是模型自评。

对应代码：

```text
code/part3-agent/
├── kernel_optimize/
│   ├── agent.py       # ReAct 主循环
│   ├── tools.py       # 三件套 + accept_candidate
│   ├── intake.py      # 从原始 kernel 建 task
│   ├── measure.py     # 峰值 + profile 载荷
│   ├── prompts.py     # SYSTEM_SOP
│   └── __main__.py    # 对话式 / --batch
├── skills/rocm-kernel-optimize/
└── chapter15/fixtures/   # 第 17 章实战入口
```



## 16.1 Agent 的整体架构

最简循环六步：

```text
读题 → 生成 → compile_kernel → bench_kernel → profile_kernel → accept_candidate
         ↑                         失败也回到生成 ←──────────────┘
```

```mermaid
flowchart LR
  U[用户 / fixtures] --> I[读题与建任务]
  I --> P[measure_peak + profile_kernel]
  P --> G[LLM 生成候选]
  G --> C[compile_kernel]
  C --> B[bench_kernel]
  B --> Pr[profile_kernel]
  Pr --> A[accept_candidate]
  A -->|接受| Best[更新 best.py]
  A -->|拒绝| R[读错因 / 换机制]
  Best --> G
  R --> G
  G --> T{终止?}
  T -->|否| G
  T -->|是| Rep[报告 + 确定性汇总]
```

| 步骤 | 输入 | 输出 |
|---|---|---|
| 读题 | task.json / fixtures | 可检查的合同理解 |
| 生成 | 合同 + best + 反馈 | 新候选源码 |
| compile | 源码 | `{ok, stage, errors}` |
| bench | 源码 | median/p95/带宽/算力 |
| profile | 源码 | bottleneck / AI / bound |
| accept | 源码 + change | accepted + 轨迹行 |

设计原则：

1. **工具结果就是事实**——LLM 不自报加速比；
2. **确定性代码掌权，LLM 受控提议**；
3. **分步三件套**；**唯一晋升入口** `accept_candidate`。



## 16.2 读题与问题理解

1. **对话式 intake**（`python -m kernel_optimize`）  
   `ask_user` 收集 kernel / shape / dtype → `get_environment` → 必要时 `convert_kernel` → `setup_task`。

2. **非交互 batch**（第 17 章主路径）  
   预置 fixtures，跳过提问，直接优化。

读题后要做两件事：

1. 把规格变成可检查条件（reference、atol/rtol、入口签名）；
2. 把性能目标量化（相对 baseline 的改进阈值，或绝对延迟）。

合同字段：入口签名、`shape`、`tensors` / `launchArguments`、正确性容差、benchmark 参数、`optimization.minImprovementFraction`。



## 16.3 生成初始 kernel

第一版目标只有一个：**正确**。它不追求快，但要成为之后所有比较的基线。

- baseline：用户粘贴，或 fixtures 里故意偏小 `block_size` 的朴素实现；
- 候选：LLM 根据 `profile_kernel` 与 `read_reference("optimization-patterns")`，一次改**一个主要机制**；
- 非 Triton：先 `convert_kernel`，再进入 compile / bench / accept。

第 16 章 `vector_add` fixtures 的 baseline 使用 `block_size=256`、未设 `num_warps`——给 Agent 留出「减 grid 启动开销 / 提高并行度」的可搜索头寸。



## 16.4 跑分与性能反馈

每一轮标准动作：

1. `compile_kernel(source)` — 不过则修源码；
2. `bench_kernel(source)` — 看 mean/median/p95/带宽/算力；
3. `profile_kernel(source)` — 首轮与换机制时必须；
4. `accept_candidate(source, change)` — 配对裁决；接受则覆盖 `best.py`；无论成败追加 `trajectory.jsonl`。

拿到数字后要分层比较：

1. 和合同目标比（是否值得继续）；
2. 和上一版 incumbent 比（提升 / 回退 / 噪声）；
3. 改进幅度小于阈值 → 视为无变化（`below_threshold`）。

辅助：`measure_peak` 给 Roofline 天花板；单独 `bench_kernel` 可摸底，但晋升仍以 `accept_candidate` 为准。



## 16.5 反思与改写

一次好的反思包含三个层次：

1. **现象**：数字是多少，对比目标差多少；
2. **原因**：profiling 说了什么——访存受限？占用不足？启动开销？
3. **行动**：下一步改什么，预期是什么。

映射到工具观察：

- 读 `compile_kernel` 的 `errors[]` → 修语法；
- 读 `accept_candidate` 的 `below_threshold` → 换方向；
- 读 `profile_kernel` 的 `bottleneck` → memory-bound 减搬运/减启动，compute-bound 减运算。

完成护栏：没进入闭环（未调用 compile/bench/accept）就输出最终文本，会被 nudge 回来。

还要会**承认方向走不通**：第 16 章 R5 把 `block_size` 推到 8192 反而大幅变慢，Agent 拒绝后回到 4096 路线——失败记录与成功同等重要。



## 16.6 迭代终止条件

| 条件 | 行为 |
|---|---|
| 模型不再调工具，且已进入过闭环 | 输出最终中文报告 |
| `max_steps` 触顶 | 返回步数上限提示，仍打印确定性汇总 |
| patience / 连续无提升 | SOP 引导主动收尾 |
| 阻塞显式化 | 剩余问题不再是「优化」（环境/规格做不到） |

入口结束后打印基于 `trajectory.jsonl` 的确定性汇总——报告的账本是轨迹，不是模型口头数字。

```bash
cd code/part3-agent
uv sync && source ./activate-rocm.sh
uv run python -m kernel_optimize
uv run python -m kernel_optimize --batch chapter15/fixtures/vector_add
```



## Execution

### 步骤1：进入 Token Factory

如果尚未进入平台，先阅读项目内的 [AMD Radeon Cloud 使用教程](../../docs/cloud/amd-radeon-cloud/index.md)，按照其中的 **“立即体验” → “注册与登录” → “进入云算力 / 创建工作区”** 完成登录并进入 AMD 开发者云。

进入 Radeon Cloud 后，模型 API 入口位于页面顶部导航栏：点击 **Gallery** 右侧的 **Token Factory**。

![进入 Token Factory](./images/radeon-cloud-open-token-factory.png)


在 **Public Free Model APIs** 中选择 `DeepSeek-V4-Flash-0731`。

![选择 DeepSeek-V4-Flash-0731](./images/radeon-cloud-select-free-model.png)


打开模型卡片，确认以下信息：

| 项目 | 值 |
|---|---|
| Base URI | `https://developer.amd.com.cn/radeon/api/v1` |
| Model | `DeepSeek-V4-Flash` |

点击 API Key 右侧的复制按钮。后面的输入单元会接收这个 Key，输入内容不会显示。

![复制 API Key](./images/radeon-cloud-copy-api-key.png)

> 如使用 `Qwen3.6-35B-A3B`，打开对应模型卡片并复制其中显示的 Model 与 API Key，步骤相同。


### 步骤2：定位仓库根目录


In [ ]:
from pathlib import Path
import importlib.util
import json
import subprocess
import sys


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        for repo in (candidate, candidate / "hello-gpu"):
            if (
                (repo / "code/part3-agent/kernel_optimize/agent.py").is_file()
                and (repo / "notebooks/part3-agent").is_dir()
            ):
                return repo
    raise FileNotFoundError("未找到 hello-gpu 仓库；请从仓库目录或其父目录打开本 Notebook。")


REPO_ROOT = find_repo_root()
PART3_ROOT = REPO_ROOT / "code/part3-agent"
FIXTURE_ROOT = PART3_ROOT / "chapter15/fixtures/vector_add"
if str(PART3_ROOT) not in sys.path:
    sys.path.insert(0, str(PART3_ROOT))

print(f"仓库根目录: {REPO_ROOT}")
print(f"代码目录: {PART3_ROOT}")
print(f"Python: {sys.executable}")


### 步骤3：准备 Agent 依赖

PyTorch、Triton 和 ROCm 使用云端已有环境；这里只安装 Agent 层缺失的 Python 包。


In [ ]:
requirements = {
    "litellm": "litellm>=1.50",
    "pydantic": "pydantic>=2.0",
    "prompt_toolkit": "prompt-toolkit>=3.0",
    "fastapi": "fastapi>=0.100",
    "orjson": "orjson>=3.0",
}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]

if missing:
    command = [sys.executable, "-m", "pip", "install", *missing]
    print("安装缺失依赖:", " ".join(missing))
    subprocess.run(command, check=True)
    importlib.invalidate_caches()
else:
    print("Agent 依赖已就绪")


### 步骤4：配置模型接口

默认使用 Radeon Cloud 的 DeepSeek-V4-Flash。若要改用 DeepSeek 官方 API，请在下方代码单元中注释 Radeon Cloud 的四行配置，再取消 DeepSeek 官方四行配置的注释，并粘贴对应平台的 API Key。

DeepSeek 官方参数来自 [首次调用 API](https://api-docs.deepseek.com/zh-cn/)：Base URL 为 `https://api.deepseek.com`，模型为 `deepseek-v4-flash`；LiteLLM 使用 `deepseek/` 前缀。官方 Key 在 [DeepSeek 开放平台](https://platform.deepseek.com/api_keys) 创建。官方思考模式默认开启；本 Agent 使用多轮 Tool Calls，因此配置为非思考模式，避免额外维护 `reasoning_content`。


In [ ]:
import os
from getpass import getpass

# Radeon Cloud（默认）：保留下面四行。
PROVIDER_LABEL = "Radeon Cloud"
API_BASE = "https://developer.amd.com.cn/radeon/api/v1"
MODEL = "openai/DeepSeek-V4-Flash"
EXTRA_BODY = {"enable_thinking": False}

# DeepSeek 官方：注释上面四行，再取消下面四行的注释。
# PROVIDER_LABEL = "DeepSeek 官方"
# API_BASE = "https://api.deepseek.com"
# MODEL = "deepseek/deepseek-v4-flash"
# EXTRA_BODY = {"thinking": {"type": "disabled"}}

API_KEY = getpass(f"粘贴 {PROVIDER_LABEL} API Key（输入不会显示）: ").strip()
if not API_KEY:
    raise ValueError("API Key 不能为空")

os.environ["KERNEL_AGENT_MODEL"] = MODEL
os.environ["KERNEL_AGENT_API_BASE"] = API_BASE
os.environ["KERNEL_AGENT_API_KEY"] = API_KEY
os.environ["KERNEL_AGENT_EXTRA_BODY"] = json.dumps(EXTRA_BODY)
os.environ["KERNEL_AGENT_CREDENTIAL_PROFILE"] = f"{MODEL}|{API_BASE}"
os.environ["OPENAI_API_KEY"] = API_KEY
if MODEL.startswith("deepseek/"):
    os.environ["DEEPSEEK_API_KEY"] = API_KEY

print(f"Provider: {PROVIDER_LABEL}")
print(f"Model: {MODEL}")
print(f"API Base: {API_BASE}")
print(f"Extra Body: {json.dumps(EXTRA_BODY, ensure_ascii=False)}")
print("API Key: configured")


### 步骤5：检查模型接口与请求频率

平台模型接口限制为 **20 RPM**。下面先安装 Notebook 内的请求包装器：两次模型请求至少间隔 3 秒；若公共模型仍返回 429，只对限流错误按 `5 / 10` 秒退避重试。其他异常不会被重试。

随后发送一次简短请求，确认地址、模型名和 API Key 可以正常使用。


In [ ]:
import threading
import time
from kernel_optimize import llm

MODEL_RPM_LIMIT = 20
MIN_MODEL_REQUEST_INTERVAL_S = 60.0 / MODEL_RPM_LIMIT
RATE_LIMIT_BACKOFF_S = (5, 10)

# 重跑本单元时始终保留最初的 chat，避免重复套多层包装器。
if not hasattr(llm, "_hello_gpu_original_chat"):
    llm._hello_gpu_original_chat = llm.chat
_original_llm_chat = llm._hello_gpu_original_chat
_model_request_lock = threading.Lock()
_last_model_request_started = 0.0


def is_rate_limit_error(error):
    text = str(error).lower()
    return (
        getattr(error, "status_code", None) == 429
        or error.__class__.__name__ == "RateLimitError"
        or ("429" in text and ("rate limit" in text or "rate_limit" in text))
    )


def retry_after_seconds(error):
    headers = getattr(error, "litellm_response_headers", None)
    if not headers:
        response = getattr(error, "response", None)
        headers = getattr(response, "headers", None) if response is not None else None
    if not headers:
        return None
    value = headers.get("retry-after") or headers.get("Retry-After")
    try:
        return max(0.0, float(value))
    except (TypeError, ValueError):
        return None


def rate_limited_chat(messages, **kwargs):
    global _last_model_request_started
    with _model_request_lock:
        for attempt in range(len(RATE_LIMIT_BACKOFF_S) + 1):
            elapsed = time.monotonic() - _last_model_request_started
            pacing_wait = MIN_MODEL_REQUEST_INTERVAL_S - elapsed
            if pacing_wait > 0:
                time.sleep(pacing_wait)

            _last_model_request_started = time.monotonic()
            try:
                return _original_llm_chat(messages, **kwargs)
            except Exception as error:
                if not is_rate_limit_error(error):
                    raise
                if attempt >= len(RATE_LIMIT_BACKOFF_S):
                    raise
                server_wait = retry_after_seconds(error)
                wait_seconds = max(
                    MIN_MODEL_REQUEST_INTERVAL_S,
                    server_wait if server_wait is not None else RATE_LIMIT_BACKOFF_S[attempt],
                )
                print(
                    f"模型接口 429 限流，{wait_seconds:.0f} 秒后重试 "
                    f"({attempt + 1}/{len(RATE_LIMIT_BACKOFF_S)})"
                )
                time.sleep(wait_seconds)


llm.chat = rate_limited_chat
MODEL_API_READY = False
try:
    message = llm.chat(
        [{"role": "user", "content": "请只回复 READY"}],
        temperature=0.0,
    )
except Exception as error:
    if not is_rate_limit_error(error):
        raise
    print("公共模型持续限流。请稍后重新运行本单元，再继续步骤8。")
else:
    MODEL_API_READY = True
    print(message.content)
    print(
        f"请求节流已启用：RPM={MODEL_RPM_LIMIT}，"
        f"最小间隔={MIN_MODEL_REQUEST_INTERVAL_S:.1f}s"
    )


### 步骤6：检查 GPU 环境

完整闭环需要当前 Python 能访问 ROCm GPU、PyTorch 和 Triton。


In [ ]:
import torch
import triton

if not torch.cuda.is_available():
    raise RuntimeError("当前 Python 未检测到 ROCm GPU")

properties = torch.cuda.get_device_properties(0)
gpu_arch = str(getattr(properties, "gcnArchName", "")).split(":", 1)[0]
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU arch: {gpu_arch or 'unknown'}")
print(f"ROCm: {torch.version.hip}")
print(f"PyTorch: {torch.__version__}")
print(f"Triton: {triton.__version__}")


### 步骤7：建立 batch 工作区

使用 `chapter15/fixtures/vector_add` 作为任务入口。工作区保存 task、baseline、best 和优化轨迹。


In [ ]:
from datetime import datetime, timezone
from kernel_optimize.__main__ import _seed_workspace_from_fixture

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
AGENT_WORKSPACE = REPO_ROOT / "runs/part3-agent/chapter16" / run_id
_seed_workspace_from_fixture(FIXTURE_ROOT, AGENT_WORKSPACE)

print(f"工作区: {AGENT_WORKSPACE}")
for path in sorted(AGENT_WORKSPACE.iterdir()):
    print(f"  - {path.name}")


### 步骤8：运行多轮优化 Agent

模型根据 task、baseline 和工具反馈生成候选。正确性、计时和晋升分别由 `compile_kernel`、`bench_kernel`、`profile_kernel` 和 `accept_candidate` 完成。

本步骤沿用步骤5的 20 RPM 节流与 429 退避。若两次重试后仍限流，当前工作区、`best.py` 和已写入的轨迹都会保留；等待后重新运行本步骤即可从当前 best 开始新一轮。
当前拆分工具模式使用 25 个模型步骤。batch 主循环不会向模型暴露 `run_code`；候选通过 `bench_kernel` 后，必须先用同一份 source 完成 `accept_candidate` 裁决，才能开始下一个候选。达到最大步数时，最后一个已完成 benchmark 的候选会自动提交权威裁决。
若步数耗尽时最后一个候选已经成功 benchmark，主循环会自动调用 `accept_candidate`；若它只有 compile 结果或 benchmark 未通过，则该未完成候选不计入轨迹，前面已经完成裁决的轨迹仍可继续分析，并在 `agent-status.json` 中标记为 `complete_with_warning`。


In [ ]:
from kernel_optimize.agent import run_agent

MAX_STEPS = 25


def print_step(step, action, observation):
    if action.startswith("tool:"):
        print(f"[step {step}] {action.split(':', 1)[1]}")
    elif action.startswith("result:") and observation:
        first_line = observation.strip().splitlines()[0] if observation.strip() else ""
        print(f"           {first_line[:200]}")
    elif action == "nudge":
        print(f"[step {step}] continue")
    elif action == "done":
        print(f"[step {step}] done")


report = None
AGENT_RUN_COMPLETE = False
if not MODEL_API_READY:
    print("模型接口检查尚未通过；请先重新运行步骤5。")
else:
    try:
        report = run_agent(
            AGENT_WORKSPACE,
            max_steps=MAX_STEPS,
            on_step=print_step,
            batch=True,
        )
    except Exception as error:
        if not is_rate_limit_error(error):
            raise
        print("公共模型在重试后仍返回 429，本次 Agent 暂停。")
        print(f"工作区已保留: {AGENT_WORKSPACE}")
        print("请等待平台限流恢复后重新运行步骤8；不需要重跑前面的 GPU 检查。")
    else:
        status_path = AGENT_WORKSPACE / "agent-status.json"
        agent_status = (
            json.loads(status_path.read_text(encoding="utf-8"))
            if status_path.is_file()
            else {}
        )
        AGENT_RUN_COMPLETE = agent_status.get("evidenceReady") is True
        if agent_status.get("state") == "complete_with_warning":
            print("最后一个候选未完成，已忽略该候选；已有权威轨迹仍可分析。")
        (AGENT_WORKSPACE / "agent-report.md").write_text(
            report + "\n", encoding="utf-8"
        )
        print("\n═══ Agent Report ═══")
        print(report)
        if not AGENT_RUN_COMPLETE:
            print("本轮未形成完整权威闭环；步骤9只展示已存在的工作区证据。")


### 步骤9：查看优化轨迹

最终结果以 `trajectory.jsonl` 为准。每一行对应一次候选评测，包含改动、延迟、改进比例和晋升结果。


In [ ]:
trajectory_path = AGENT_WORKSPACE / "trajectory.jsonl"
if not trajectory_path.exists():
    print("尚未生成 trajectory.jsonl")
else:
    rows = []
    malformed_rows = 0
    for line in trajectory_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            malformed_rows += 1
            continue
        if isinstance(row, dict):
            rows.append(row)

    accepted = sum(bool(row.get("accepted")) for row in rows)
    print(f"评测轮次: {len(rows)}")
    print(f"接受轮次: {accepted}")
    if malformed_rows:
        print(f"忽略损坏轨迹行: {malformed_rows}")
    for index, row in enumerate(rows, 1):
        status = "ACCEPT" if row.get("accepted") else "REJECT"
        print(
            f"[{index:02d}] {status} | latency={row.get('latencyMs')} ms | "
            f"improvement={row.get('improvementFraction')} | {row.get('change', '')}"
        )

print("\n运行产物:")
for label, artifact_path in (
    ("best.py", AGENT_WORKSPACE / "best.py"),
    ("trajectory.jsonl", trajectory_path),
    ("agent-report.md", AGENT_WORKSPACE / "agent-report.md"),
    ("agent-status.json", AGENT_WORKSPACE / "agent-status.json"),
):
    state = str(artifact_path) if artifact_path.exists() else "未生成"
    print(f"  {label}: {state}")


## Expected Output / Interpretation

1. 模型接口检查返回 `READY`；
2. 环境单元打印当前 GPU、架构、ROCm、PyTorch 和 Triton 版本；
3. Agent 运行时依次出现工具调用及其结构化返回；
4. 工作区生成 `best.py`、`trajectory.jsonl` 和 `agent-report.md`。

被拒绝的候选不会覆盖 `best.py`。达到最大步数时，已有轨迹仍然有效。

若出现 `429 / process_concurrency_rate_limit_exceeded`，Notebook 会自动限速并退避重试。重试耗尽时保留当前工作区；等待平台恢复后，只需重新运行 Agent 步骤，不必重跑已完成的 GPU 工具。

`trajectory.jsonl` 只由 `accept_candidate` 创建。若没有任何候选完成 benchmark，Agent 会明确报告“未生成轨迹”，不会再输出一个不存在的文件路径。更新 `code/` 后若当前 kernel 仍缓存旧模块，请重启 kernel，再从模型配置步骤运行到步骤8。

当最大步数恰好停在新候选的 compile 阶段时，该未完成候选不会写入轨迹，也不会影响前面已完成的接受/拒绝记录；后续分析只使用已有权威轨迹。


## Pass Criteria

1. 模型能够返回 function calling；
2. Agent 至少进入一次 compile / bench / accept 闭环；
3. 每次候选评测都写入 `trajectory.jsonl`；
4. 报告中的性能结论能够在轨迹中找到对应记录。


## 本章小结

- 算子优化 Agent = 短主循环 + 线上对齐三件套 + `accept_candidate` + SOP。
- 读题 → 测峰值/profiling → 生成 → compile/bench/profile → accept → 反思。
- 数字永不口算；晋升只认配对证据；终止条件要显式。



## 延伸阅读

- `code/part3-agent/REFACTOR-PLAN-v2.md`
- `code/part3-agent/skills/rocm-kernel-optimize/SKILL.md`
- 下一章：多轮实战与可视化报告
